# MQTT Broker

## Initial Setup

This notebook documents the development process of creating the self hosted cloud core MQTT broker.

To begin, we define a minimal mosquitto.conf to setup our broker config. This will set the listening port, disable anonymous access and point to a password file.

```conf
# Disable anonymous access
allow_anonymous false

# Set password file
password_file /mosquitto/config/passwd

# Default listener
listener 1883

```

This version of the broker will be self hosted. To facilitate this we will use Docker to containerise the MQTT broker. We will use the existing `eclipse-mosquitto` image and a yml file to orchestrate the broker and its various connections to future images.

The important parts to note is the mounted port,
```yaml
# Map port 1883 from the container to the host
# This allows MQTT clients to connect to localhost:1883 on the host
ports:
    - "1883:1883"
```
and the mounted volumes,
```yaml
# Mount configuration files into the container
volumes:
    # Mount custom Mosquitto config file from host to container path
    - ./broker/mosquitto.conf:/mosquitto/config/mosquitto.conf

    # Mount password file for authentication
    - ./broker/passwd:/mosquitto/config/passwd
```
this allows us to point to our generated password hash and config.

Next we need to set our user name and password, we can do this through a docker command:
```bash
docker run --rm -v "$PWD/broker:/mosquitto/config" eclipse-mosquitto mosquitto_passwd -b -c /mosquitto/config/passwd smartuser testpwd
```

This will create the username 'smartuser' with the password 'testpwd'.

This is the minimal needed setup to start the broker service. We can launch the broker using:
```bash
docker-compose up -d

```
and then inspect the logs to make sure that we are up
```bash
docker logs mqtt-broker
```

To be extra sure that we are running as expected, we can use 
```bash
mosquitto_sub -h localhost -t test/topic -u smartuser -P testpwd
```
in one terminal, then in a seperate terminal
```bash
mosquitto_pub -h localhost -t test/topic -m "hello from docker mqtt" -u smartuser -P testpwd
```
which will pulish a message in the original terminal.

# TLS Encryption

For added security, we will add TLS encryption to MQTT Broker. Since this is a development (or demo) environment, we will use self signed certificates. The setup and encryption are the same, however in production there will be a few differences in certificate use:
- We will use self signed certs, production TLS will use publicly trusted certificiates.
- Our trust chain will be manually accepted by clients, whereas production will be handled by the OS.
- Our clients will be configured to trust the CA, whereas production TLS trust will be automatic.



To begin, we will start by generating CA certificates. We will `openssl`'s `genrsa` commands for simplicity. This is largely boilerplate to make development simpler,
```bash
# Generate a private key for the Certificate Authority
echo "Generating CA private key and certificate..."
openssl genrsa -out ca.key 2048
```
the full script is available in `broker/generate_certs.sh`.

Now we need to update the mosquitto config to allow TLS listeners,
```conf
# Plain MQTT (optional for fallback)
listener 1883

# TLS MQTT
listener 8883
cafile /mosquitto/config/certs/ca.crt
certfile /mosquitto/config/certs/server.crt
keyfile /mosquitto/config/certs/server.key
```
this will open a new port for TLS connections and point to the certificate files. We will keep the plain listener for now just in case it turns out to be useful, it won't be a lot of work to remove.

Finally, we need to include the new TLS port and certificate volumes in the docker file. 
```yaml
ports:
- "1883:1883"   # Plain MQTT (optional)
- "8883:8883"   # Secure MQTT

# Mount configuration files into the container
volumes:
# Mount custom Mosquitto config file from host to container path
- ./broker/mosquitto.conf:/mosquitto/config/mosquitto.conf

# Mount password file for authentication
- ./broker/passwd:/mosquitto/config/passwd

# Mount self CA certificates
- ./broker/certs:/mosquitto/config/certs
```

We can test this connection in the same way we tested plain MQTT, by composing the docker and publishing a message to the TLS port.
```bash
# In one terminal
mosquitto_sub -h localhost -t test/topic -u smartuser -P testpwd -p 8883 \
 --cafile broker/certs/ca.crt

# In another terminal
mosquitto_pub -h localhost -t test/topic -m "Hello with TLS" -u smartuser -P testpwd -p 8883 \
 --cafile broker/certs/ca.crt
```

For simplicity, there is a `start_broker.sh` script to start the broker docker cleanly and print the logs. This can be called through the Makefile by using one of the commands in there.
```bash
# E.g.
make start
```